In [1]:
pip install jupyter ipykernel --break-system-packages

  Using cached fqdn-1.5.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached isoduration-20.11.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached uri_template-1.3.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached webcolors-25.10.0-py3-none-any.whl.metadata (2.2 kB)
Using cached webcolors-25.10.0-py3-none-any.whl (14 kB)
Using cached fqdn-1.5.1-py3-none-any.whl (9.1 kB)
Using cached isoduration-20.11.0-py3-none-any.whl (11 kB)
Using cached uri_template-1.3.0-py3-none-any.whl (11 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import pandas as pd

# Path to your raw IPL data
ipl_path = "../data/raw/ipl"

# List all JSON files in the folder
files = [f for f in os.listdir(ipl_path) if f.endswith('.json')]

print(f"Number of IPL match files: {len(files)}")
print(f"Sample filenames: {files[:5]}")

Number of IPL match files: 1169
Sample filenames: ['1426261.json', '1359507.json', '392217.json', '1254112.json', '829817.json']


In [3]:
# Open one match file and inspect the structure
sample_file = os.path.join(ipl_path, files[0])

with open(sample_file) as f:
    match = json.load(f)

# Look at the top level keys
print("Top level keys:", match.keys())

Top level keys: dict_keys(['meta', 'info', 'innings'])


In [4]:
import json

print("META:")
print(json.dumps(match['meta'], indent=2))

print("\nINFO:")
print(json.dumps(match['info'], indent=2))

META:
{
  "data_version": "1.1.0",
  "created": "2024-04-09",
  "revision": 1
}

INFO:
{
  "balls_per_over": 6,
  "city": "Mohali",
  "dates": [
    "2024-04-09"
  ],
  "event": {
    "name": "Indian Premier League",
    "match_number": 23
  },
  "gender": "male",
  "match_type": "T20",
  "officials": {
    "match_referees": [
      "V Narayan Kutty"
    ],
    "reserve_umpires": [
      "PM Joshi"
    ],
    "tv_umpires": [
      "AG Wharf"
    ],
    "umpires": [
      "Navdeep Singh",
      "NA Patwardhan"
    ]
  },
  "outcome": {
    "winner": "Sunrisers Hyderabad",
    "by": {
      "runs": 2
    }
  },
  "overs": 20,
  "player_of_match": [
    "Nithish Kumar Reddy"
  ],
  "players": {
    "Sunrisers Hyderabad": [
      "TM Head",
      "Abhishek Sharma",
      "AK Markram",
      "Nithish Kumar Reddy",
      "RA Tripathi",
      "H Klaasen",
      "Abdul Samad",
      "Shahbaz Ahmed",
      "PJ Cummins",
      "B Kumar",
      "JD Unadkat",
      "T Natarajan"
    ],
    "Punjab

In [5]:
print("INNINGS keys:", match['innings'][0].keys())
print("\nFirst over, first ball:")
print(json.dumps(match['innings'][0]['overs'][0], indent=2))

INNINGS keys: dict_keys(['team', 'overs', 'powerplays'])

First over, first ball:
{
  "over": 0,
  "deliveries": [
    {
      "batter": "TM Head",
      "bowler": "K Rabada",
      "non_striker": "Abhishek Sharma",
      "runs": {
        "batter": 0,
        "extras": 0,
        "total": 0
      }
    },
    {
      "batter": "TM Head",
      "bowler": "K Rabada",
      "non_striker": "Abhishek Sharma",
      "runs": {
        "batter": 0,
        "extras": 0,
        "total": 0
      }
    },
    {
      "batter": "TM Head",
      "bowler": "K Rabada",
      "non_striker": "Abhishek Sharma",
      "runs": {
        "batter": 4,
        "extras": 0,
        "total": 4
      }
    },
    {
      "batter": "TM Head",
      "bowler": "K Rabada",
      "non_striker": "Abhishek Sharma",
      "runs": {
        "batter": 0,
        "extras": 0,
        "total": 0
      }
    },
    {
      "batter": "TM Head",
      "bowler": "K Rabada",
      "non_striker": "Abhishek Sharma",
      "runs"

In [6]:
def parse_match(filepath):
    with open(filepath) as f:
        match = json.load(f)
    
    info = match['info']
    rows = []
    
    for innings_num, innings in enumerate(match['innings']):
        team_batting = innings['team']
        
        for over_data in innings['overs']:
            over = over_data['over']
            
            for ball_num, delivery in enumerate(over_data['deliveries']):
                row = {
                    # Match level
                    'match_id': os.path.basename(filepath).replace('.json', ''),
                    'date': info['dates'][0],
                    'venue': info.get('venue', None),
                    'city': info.get('city', None),
                    'season': info.get('season', None),
                    'match_number': info.get('event', {}).get('match_number', None),
                    'winner': info.get('outcome', {}).get('winner', None),
                    
                    # Innings level
                    'innings': innings_num + 1,
                    'batting_team': team_batting,
                    'bowling_team': [t for t in info['teams'] if t != team_batting][0],
                    
                    # Ball level
                    'over': over,
                    'ball': ball_num + 1,
                    'batter': delivery['batter'],
                    'bowler': delivery['bowler'],
                    'non_striker': delivery['non_striker'],
                    
                    # Runs
                    'runs_batter': delivery['runs']['batter'],
                    'runs_extras': delivery['runs']['extras'],
                    'runs_total': delivery['runs']['total'],
                    
                    # Extras
                    'wide': delivery.get('extras', {}).get('wides', 0),
                    'no_ball': delivery.get('extras', {}).get('noballs', 0),
                    'bye': delivery.get('extras', {}).get('byes', 0),
                    'leg_bye': delivery.get('extras', {}).get('legbyes', 0),
                    
                    # Wicket
                    'wicket': 1 if 'wickets' in delivery else 0,
                    'wicket_kind': delivery.get('wickets', [{}])[0].get('kind', None),
                    'player_out': delivery.get('wickets', [{}])[0].get('player_out', None),
                }
                rows.append(row)
    
    return rows

# Test on one match
rows = parse_match(sample_file)
df_sample = pd.DataFrame(rows)
print(f"Shape: {df_sample.shape}")
print(df_sample.head())

Shape: (251, 25)
  match_id        date                                              venue  \
0  1426261  2024-04-09  Maharaja Yadavindra Singh International Cricke...   
1  1426261  2024-04-09  Maharaja Yadavindra Singh International Cricke...   
2  1426261  2024-04-09  Maharaja Yadavindra Singh International Cricke...   
3  1426261  2024-04-09  Maharaja Yadavindra Singh International Cricke...   
4  1426261  2024-04-09  Maharaja Yadavindra Singh International Cricke...   

     city season  match_number               winner  innings  \
0  Mohali   2024            23  Sunrisers Hyderabad        1   
1  Mohali   2024            23  Sunrisers Hyderabad        1   
2  Mohali   2024            23  Sunrisers Hyderabad        1   
3  Mohali   2024            23  Sunrisers Hyderabad        1   
4  Mohali   2024            23  Sunrisers Hyderabad        1   

          batting_team  bowling_team  ...  runs_batter  runs_extras  \
0  Sunrisers Hyderabad  Punjab Kings  ...            0          

In [7]:
df_sample

,match_id,date,venue,city,season,match_number,winner,innings,batting_team,bowling_team,...,runs_batter,runs_extras,runs_total,wide,no_ball,bye,leg_bye,wicket,wicket_kind,player_out
0,1426261,2024-04-09,Maharaja Yadavindra Singh International Cricke...,Mohali,2024,23,Sunrisers Hyderabad,1,Sunrisers Hyderabad,Punjab Kings,...,0,0,0,0,0,0,0,0,None,None
1,1426261,2024-04-09,Maharaja Yadavindra Singh International Cricke...,Mohali,2024,23,Sunrisers Hyderabad,1,Sunrisers Hyderabad,Punjab Kings,...,0,0,0,0,0,0,0,0,None,None
2,1426261,2024-04-09,Maharaja Yadavindra Singh International Cricke...,Mohali,2024,23,Sunrisers Hyderabad,1,Sunrisers Hyderabad,Punjab Kings,...,4,0,4,0,0,0,0,0,None,None
3,1426261,2024-04-09,Maharaja Yadavindra Singh International Cricke...,Mohali,2024,23,Sunrisers Hyderabad,1,Sunrisers Hyderabad,Punjab Kings,...,0,0,0,0,0,0,0,0,None,None
4,1426261,2024-04-09,Maharaja Yadavindra Singh International Cricke...,Mohali,2024,23,Sunrisers Hyderabad,1,Sunrisers Hyderabad,Punjab Kings,...,0,0,0,0,0,0,0,0,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246,1426261,2024-04-09,Maharaja Yadavindra Singh International Cricke...,Mohali,2024,23,Sunrisers Hyderabad,2,Punjab Kings,Sunrisers Hyderabad,...,2,0,2,0,0,0,0,0,None,None
247,1426261,2024-04-09,Maharaja Yadavindra Singh International Cricke...,Mohali,2024,23,Sunrisers Hyderabad,2,Punjab Kings,Sunrisers Hyderabad,...,2,0,2,0,0,0,0,0,None,None
248,1426261,2024-04-09,Maharaja Yadavindra Singh International Cricke...,Mohali,2024,23,Sunrisers Hyderabad,2,Punjab Kings,Sunrisers Hyderabad,...,0,1,1,1,0,0,0,0,None,None
249,1426261,2024-04-09,Maharaja Yadavindra Singh International Cricke...,Mohali,2024,23,Sunrisers Hyderabad,2,Punjab Kings,Sunrisers Hyderabad,...,1,0,1,0,0,0,0,0,None,None


In [8]:
from tqdm import tqdm

all_rows = []

for filename in tqdm(files):
    filepath = os.path.join(ipl_path, filename)
    try:
        rows = parse_match(filepath)
        all_rows.extend(rows)
    except Exception as e:
        print(f"Error parsing {filename}: {e}")

df_ipl = pd.DataFrame(all_rows)
print(f"\nFinal shape: {df_ipl.shape}")
print(f"Date range: {df_ipl['date'].min()} to {df_ipl['date'].max()}")
print(f"Total deliveries: {len(df_ipl)}")

100%|██████████| 1169/1169 [00:01<00:00, 828.12it/s]



Final shape: (278205, 25)
Date range: 2008-04-18 to 2025-06-03
Total deliveries: 278205


In [9]:
# Save to processed folder
output_path = "../data/processed/ipl_deliveries.csv"
df_ipl.to_csv(output_path, index=False)
print(f"Saved to {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB")

Saved to ../data/processed/ipl_deliveries.csv
File size: 47.8 MB


In [10]:
# Sanity checks
print("=== BASIC STATS ===")
print(f"Unique matches: {df_ipl['match_id'].nunique()}")
print(f"Unique batters: {df_ipl['batter'].nunique()}")
print(f"Unique bowlers: {df_ipl['bowler'].nunique()}")
print(f"Unique teams: {df_ipl['batting_team'].nunique()}")
print(f"Seasons covered: {sorted(df_ipl['season'].unique(), key=str)}")

print("\n=== RUNS DISTRIBUTION ===")
print(df_ipl['runs_total'].value_counts().sort_index())

print("\n=== WICKET TYPES ===")
print(df_ipl['wicket_kind'].value_counts())

=== BASIC STATS ===
Unique matches: 1169
Unique batters: 703
Unique bowlers: 550
Unique teams: 19
Seasons covered: ['2007/08', 2009, '2009/10', 2011, 2012, '2012', 2013, '2013', 2014, 2015, '2015', 2016, '2016', 2017, '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025']

=== RUNS DISTRIBUTION ===
runs_total
0     95778
1    115711
2     18349
3       954
4     32502
5       557
6     14259
7        95
Name: count, dtype: int64

=== WICKET TYPES ===
wicket_kind
caught                   8665
bowled                   2345
run out                  1153
lbw                       853
caught and bowled         388
stumped                   376
hit wicket                 18
retired hurt               17
retired out                 5
obstructing the field       3
Name: count, dtype: int64


In [11]:
# Clean season column
def clean_season(season):
    season = str(season)
    # Convert "2007/08" format to just the starting year
    if '/' in season:
        return int(season.split('/')[0])
    return int(season)

df_ipl['season'] = df_ipl['season'].apply(clean_season)

print("Cleaned seasons:", sorted(df_ipl['season'].unique()))
print("Unique teams:", sorted(df_ipl['batting_team'].unique()))

Cleaned seasons: [2007, 2009, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Unique teams: ['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Delhi Daredevils', 'Gujarat Lions', 'Gujarat Titans', 'Kings XI Punjab', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Rising Pune Supergiants', 'Royal Challengers Bangalore', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


In [12]:
# Standardise team names
team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
}

df_ipl['batting_team'] = df_ipl['batting_team'].replace(team_mapping)
df_ipl['bowling_team'] = df_ipl['bowling_team'].replace(team_mapping)

print("Unique teams after cleaning:", sorted(df_ipl['batting_team'].unique()))
print(f"Total teams: {df_ipl['batting_team'].nunique()}")

Unique teams after cleaning: ['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiants', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']
Total teams: 15


In [13]:
df_ipl.to_csv("../data/processed/ipl_deliveries.csv", index=False)
print("Saved cleaned dataframe")

Saved cleaned dataframe


In [14]:
# Check Sachin's 2009 season specifically
sachin_2009 = df_ipl[(df_ipl['batter'] == 'SR Tendulkar') & (df_ipl['season'] == 2009)]

print(f"Total deliveries faced: {len(sachin_2009)}")
print(f"Total runs (runs_batter): {sachin_2009['runs_batter'].sum()}")
print(f"Wides in his rows: {sachin_2009['wide'].sum()}")
print(f"No balls in his rows: {sachin_2009['no_ball'].sum()}")
print(f"\nRuns breakdown:")
print(sachin_2009['runs_batter'].value_counts().sort_index())

Total deliveries faced: 804
Total runs (runs_batter): 982
Wides in his rows: 50
No balls in his rows: 7

Runs breakdown:
runs_batter
0    319
1    295
2     47
3      5
4    125
6     13
Name: count, dtype: int64


In [15]:
def clean_season(season):
    season = str(season)
    if '/' in season:
        # "2009/10" → take the second part → 2010
        # "2007/08" → take the second part → 2008
        second = season.split('/')[1]
        # Handle both "08" and "10" formats
        if len(second) == 2:
            prefix = season.split('/')[0][:2]
            return int(prefix + second)
        return int(second)
    return int(season)

# Test it
test_cases = ['2007/08', '2009/10', '2012', 2013]
for t in test_cases:
    print(f"{t} → {clean_season(t)}")

2007/08 → 2008
2009/10 → 2010
2012 → 2012
2013 → 2013


In [16]:
# Reload from raw to avoid any previous cleaning contamination
all_rows = []

for filename in tqdm(files):
    filepath = os.path.join(ipl_path, filename)
    try:
        rows = parse_match(filepath)
        all_rows.extend(rows)
    except Exception as e:
        print(f"Error parsing {filename}: {e}")

df_ipl = pd.DataFrame(all_rows)

# Apply all cleaning in one go
df_ipl['season'] = df_ipl['season'].apply(clean_season)

team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
}
df_ipl['batting_team'] = df_ipl['batting_team'].replace(team_mapping)
df_ipl['bowling_team'] = df_ipl['bowling_team'].replace(team_mapping)

print("Seasons:", sorted(df_ipl['season'].unique()))
print("Shape:", df_ipl.shape)

100%|██████████| 1169/1169 [00:01<00:00, 879.22it/s]


Seasons: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024, 2025]
Shape: (278205, 25)


In [17]:
# Top run scorer per season
run_scorers = (df_ipl.groupby(['season', 'batter'])['runs_batter']
               .sum()
               .reset_index()
               .sort_values(['season', 'runs_batter'], ascending=[True, False])
               .groupby('season')
               .first()
               .reset_index())

print("=== TOP RUN SCORER PER SEASON ===")
print(run_scorers[['season', 'batter', 'runs_batter']].to_string(index=False))

=== TOP RUN SCORER PER SEASON ===
 season          batter  runs_batter
   2008        SE Marsh          616
   2009       ML Hayden          572
   2010    SR Tendulkar          618
   2011        CH Gayle          608
   2012        CH Gayle          733
   2013      MEK Hussey          733
   2014      RV Uthappa          660
   2015       DA Warner          562
   2016         V Kohli          973
   2017       DA Warner          641
   2018   KS Williamson          735
   2019       DA Warner          692
   2021        KL Rahul         1302
   2022      JC Buttler          863
   2023    Shubman Gill          890
   2024         V Kohli          741
   2025 B Sai Sudharsan          759


In [18]:
# Top wicket taker per season
wicket_takers = (df_ipl[df_ipl['wicket'] == 1]
                 .groupby(['season', 'bowler'])['wicket']
                 .sum()
                 .reset_index()
                 .sort_values(['season', 'wicket'], ascending=[True, False])
                 .groupby('season')
                 .first()
                 .reset_index())

print("=== TOP WICKET TAKER PER SEASON ===")
print(wicket_takers[['season', 'bowler', 'wicket']].to_string(index=False))

=== TOP WICKET TAKER PER SEASON ===
 season            bowler  wicket
   2008     Sohail Tanvir      24
   2009          RP Singh      26
   2010           PP Ojha      22
   2011        SL Malinga      30
   2012          M Morkel      30
   2013          DJ Bravo      34
   2014         MM Sharma      26
   2015          DJ Bravo      28
   2016           B Kumar      24
   2017           B Kumar      28
   2018            AJ Tye      28
   2019          K Rabada      29
   2021         JJ Bumrah      52
   2022         YS Chahal      29
   2023         MM Sharma      31
   2024          HV Patel      30
   2025 M Prasidh Krishna      26


In [19]:
print(df_ipl[df_ipl['season'] == 2021]['match_id'].nunique())
print(df_ipl[df_ipl['season'] == 2020]['match_id'].nunique())

120
0


In [20]:
def clean_season(season):
    season = str(season)
    if '/' in season:
        # Special case - Covid IPL was played in 2020
        if season == '2020/21':
            return 2020
        second = season.split('/')[1]
        if len(second) == 2:
            prefix = season.split('/')[0][:2]
            return int(prefix + second)
        return int(second)
    return int(season)

# Test
test_cases = ['2007/08', '2009/10', '2020/21', '2012', 2013]
for t in test_cases:
    print(f"{t} → {clean_season(t)}")

2007/08 → 2008
2009/10 → 2010
2020/21 → 2020
2012 → 2012
2013 → 2013


In [21]:
all_rows = []

for filename in tqdm(files):
    filepath = os.path.join(ipl_path, filename)
    try:
        rows = parse_match(filepath)
        all_rows.extend(rows)
    except Exception as e:
        print(f"Error parsing {filename}: {e}")

df_ipl = pd.DataFrame(all_rows)

# Apply all cleaning
df_ipl['season'] = df_ipl['season'].apply(clean_season)

team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
}
df_ipl['batting_team'] = df_ipl['batting_team'].replace(team_mapping)
df_ipl['bowling_team'] = df_ipl['bowling_team'].replace(team_mapping)

# Verify
print("Seasons:", sorted(df_ipl['season'].unique()))
print("Shape:", df_ipl.shape)
print("\nMatches per season:")
print(df_ipl.groupby('season')['match_id'].nunique().to_string())

100%|██████████| 1169/1169 [00:01<00:00, 880.50it/s]


Seasons: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Shape: (278205, 25)

Matches per season:
season
2008    58
2009    57
2010    60
2011    73
2012    74
2013    76
2014    60
2015    59
2016    60
2017    59
2018    60
2019    60
2020    60
2021    60
2022    74
2023    74
2024    71
2025    74


In [22]:
run_scorers_clean = (df_ipl[df_ipl['wide'] == 0]
                     .groupby(['season', 'batter'])['runs_batter']
                     .sum()
                     .reset_index()
                     .sort_values(['season', 'runs_batter'], ascending=[True, False])
                     .groupby('season')
                     .first()
                     .reset_index())

print("=== TOP RUN SCORER PER SEASON ===")
print(run_scorers_clean[['season', 'batter', 'runs_batter']].to_string(index=False))

=== TOP RUN SCORER PER SEASON ===
 season          batter  runs_batter
   2008        SE Marsh          616
   2009       ML Hayden          572
   2010    SR Tendulkar          618
   2011        CH Gayle          608
   2012        CH Gayle          733
   2013      MEK Hussey          733
   2014      RV Uthappa          660
   2015       DA Warner          562
   2016         V Kohli          973
   2017       DA Warner          641
   2018   KS Williamson          735
   2019       DA Warner          692
   2020        KL Rahul          676
   2021      RD Gaikwad          635
   2022      JC Buttler          863
   2023    Shubman Gill          890
   2024         V Kohli          741
   2025 B Sai Sudharsan          759


In [23]:
# Top wicket taker per season - excluding run outs
wicket_takers_clean = (df_ipl[
    (df_ipl['wicket'] == 1) & 
    (df_ipl['wicket_kind'] != 'run out') &
    (df_ipl['wicket_kind'] != 'retired hurt') &
    (df_ipl['wicket_kind'] != 'retired out') &
    (df_ipl['wicket_kind'] != 'obstructing the field')
    ]
    .groupby(['season', 'bowler'])['wicket']
    .sum()
    .reset_index()
    .sort_values(['season', 'wicket'], ascending=[True, False])
    .groupby('season')
    .first()
    .reset_index())

print("=== TOP WICKET TAKER PER SEASON ===")
print(wicket_takers_clean[['season', 'bowler', 'wicket']].to_string(index=False))

=== TOP WICKET TAKER PER SEASON ===
 season            bowler  wicket
   2008     Sohail Tanvir      22
   2009          RP Singh      23
   2010           PP Ojha      21
   2011        SL Malinga      28
   2012          M Morkel      25
   2013          DJ Bravo      32
   2014         MM Sharma      23
   2015          DJ Bravo      26
   2016           B Kumar      23
   2017           B Kumar      26
   2018            AJ Tye      24
   2019       Imran Tahir      26
   2020          K Rabada      32
   2021          HV Patel      32
   2022         YS Chahal      27
   2023    Mohammed Shami      28
   2024          HV Patel      24
   2025 M Prasidh Krishna      25


In [24]:
rabada_2020 = df_ipl[
    (df_ipl['bowler'] == 'K Rabada') & 
    (df_ipl['season'] == 2020) &
    (df_ipl['wicket'] == 1) &
    (df_ipl['wicket_kind'] != 'run out') &
    (df_ipl['wicket_kind'] != 'retired hurt') &
    (df_ipl['wicket_kind'] != 'retired out')
]

print(f"Total wickets: {len(rabada_2020)}")
print(f"Unique matches: {rabada_2020['match_id'].nunique()}")
print(f"\nWicket types:")
print(rabada_2020['wicket_kind'].value_counts())
print(f"\nWickets per match:")
print(rabada_2020.groupby('match_id')['wicket'].sum().sort_values(ascending=False))

Total wickets: 32
Unique matches: 14

Wicket types:
wicket_kind
caught    27
bowled     5
Name: count, dtype: int64

Wickets per match:
match_id
1216493    4
1216519    4
1237180    4
1216500    3
1216539    3
1216497    2
1216505    2
1216529    2
1216532    2
1216546    2
1216509    1
1216515    1
1216543    1
1237181    1
Name: wicket, dtype: int64


In [25]:
print("=== TOP WICKET TAKER PER SEASON ===")
print(wicket_takers_clean[['season', 'bowler', 'wicket']].to_string(index=False))

=== TOP WICKET TAKER PER SEASON ===
 season            bowler  wicket
   2008     Sohail Tanvir      22
   2009          RP Singh      23
   2010           PP Ojha      21
   2011        SL Malinga      28
   2012          M Morkel      25
   2013          DJ Bravo      32
   2014         MM Sharma      23
   2015          DJ Bravo      26
   2016           B Kumar      23
   2017           B Kumar      26
   2018            AJ Tye      24
   2019       Imran Tahir      26
   2020          K Rabada      32
   2021          HV Patel      32
   2022         YS Chahal      27
   2023    Mohammed Shami      28
   2024          HV Patel      24
   2025 M Prasidh Krishna      25


In [26]:
rabada_2020_matches = df_ipl[
    (df_ipl['bowler'] == 'K Rabada') & 
    (df_ipl['season'] == 2020) &
    (df_ipl['wicket'] == 1) &
    (~df_ipl['wicket_kind'].isin(['run out', 'retired hurt', 'retired out']))
].groupby('match_id').agg(
    wickets=('wicket', 'sum'),
    date=('date', 'first')
).sort_values('date').reset_index()

print(f"Total matches: {len(rabada_2020_matches)}")
print(f"Total wickets: {rabada_2020_matches['wickets'].sum()}")
print(rabada_2020_matches.to_string(index=False))

Total matches: 14
Total wickets: 32
match_id  wickets       date
 1216493        4 2020-09-20
 1216539        3 2020-09-25
 1216532        2 2020-09-29
 1216515        1 2020-10-03
 1216519        4 2020-10-05
 1216500        3 2020-10-09
 1216529        2 2020-10-11
 1216543        1 2020-10-14
 1216509        1 2020-10-17
 1216546        2 2020-10-20
 1216497        2 2020-10-24
 1216505        2 2020-11-02
 1237180        4 2020-11-08
 1237181        1 2020-11-10


In [27]:
# Check DC matches in 2020 - how many did they play total
dc_matches = df_ipl[
    (df_ipl['season'] == 2020) & 
    ((df_ipl['batting_team'] == 'Delhi Capitals') | 
     (df_ipl['bowling_team'] == 'Delhi Capitals'))
].groupby('match_id').agg(
    date=('date', 'first'),
    total_deliveries=('ball', 'count')
).sort_values('date').reset_index()

print(f"DC matches in our data: {len(dc_matches)}")
print(dc_matches.to_string(index=False))

DC matches in our data: 17
match_id       date  total_deliveries
 1216493 2020-09-20               256
 1216539 2020-09-25               247
 1216532 2020-09-29               247
 1216515 2020-10-03               249
 1216519 2020-10-05               252
 1216500 2020-10-09               253
 1216529 2020-10-11               247
 1216543 2020-10-14               252
 1216509 2020-10-17               246
 1216546 2020-10-20               243
 1216497 2020-10-24               250
 1216524 2020-10-27               243
 1216535 2020-10-31               209
 1216505 2020-11-02               240
 1237177 2020-11-05               247
 1237180 2020-11-08               250
 1237181 2020-11-10               235


In [28]:
# Look at his 4 wicket matches in detail
for match_id in ['1216493', '1216519', '1237180']:
    print(f"\nMatch {match_id}:")
    match_data = df_ipl[
        (df_ipl['bowler'] == 'K Rabada') & 
        (df_ipl['match_id'].astype(str) == match_id) &
        (df_ipl['wicket'] == 1)
    ][['over', 'ball', 'batter', 'wicket_kind', 'innings']]
    print(match_data.to_string(index=False))


Match 1216493:
 over  ball     batter wicket_kind  innings
    6     3 GJ Maxwell      caught        2
   15     3  K Gowtham      caught        2
    0     2   KL Rahul      caught        3
    0     3   N Pooran      bowled        3

Match 1216519:
 over  ball            batter wicket_kind  innings
   13     4           V Kohli      caught        2
   15     6 Washington Sundar      caught        2
   17     1            S Dube      bowled        2
   17     3           I Udana      caught        2

Match 1237180:
 over  ball      batter wicket_kind  innings
    1     1   DA Warner      bowled        2
   18     3 Abdul Samad      caught        2
   18     4 Rashid Khan      caught        2
   18     6  SP Goswami      caught        2
